# SVM Linear — 4 Split Ratios Comparison

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import *
sns.set_palette('Blues');plt.rcParams['axes.prop_cycle']=plt.cycler(color=plt.cm.Blues(np.linspace(0.3,0.9,8)))
import warnings;warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv('../../augmented_data.csv');X=df.drop('diagnosis',axis=1);y=df['diagnosis']
print(f"Shape:{df.shape}\nClass: {df['diagnosis'].value_counts().to_dict()}")

In [ ]:
ratios = [0.9, 0.8, 0.7, 0.6]  # test_size = 1 - train_size
results = []
labels = ['90/10', '80/20', '70/30', '60/40']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('SVM Linear — Confusion Matrices for 4 Split Ratios', fontsize=16, fontweight='bold')

for idx, (ratio, label) in enumerate(zip(ratios, labels)):
    test_size = 1 - ratio
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=test_size,random_state=42,stratify=y)
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_train); X_te = sc.transform(X_test)
    
    m = SVC(kernel='linear', probability=True, random_state=42)
    m.fit(X_tr, y_train)
    y_pred = m.predict(X_te); y_proba = m.predict_proba(X_te)[:,1]
    
    acc=accuracy_score(y_test,y_pred);prec=precision_score(y_test,y_pred)
    rec=recall_score(y_test,y_pred);f1=f1_score(y_test,y_pred)
    cm=confusion_matrix(y_test,y_pred);tn,fp,fn,tp=cm.ravel()
    spec=tn/(tn+fp);auc=roc_auc_score(y_test,y_proba)
    results.append({'Ratio':label,'Acc':acc,'Prec':prec,'Rec':rec,'F1':f1,'Spec':spec,'AUC':auc})
    
    print(f"\n=== {label} ===")
    print(f"Train:{len(X_tr)} Test:{len(X_te)} | Acc:{acc:.4f} Prec:{prec:.4f} Rec:{rec:.4f} F1:{f1:.4f} Spec:{spec:.4f} AUC:{auc:.4f}")
    print(f"CM:\n{cm}")
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['B','M'], yticklabels=['B','M'], ax=axes[0,idx])
    axes[0,idx].set_title(f'{label} Split', fontweight='bold')
    axes[0,idx].set_xlabel('Predicted'); axes[0,idx].set_ylabel('Actual')
    
    n=['Acc','Prec','Rec','F1','Spec']; v=[acc,prec,rec,f1,spec]
    c=plt.cm.Blues([0.4,0.5,0.6,0.7,0.8])
    bars = axes[1,idx].bar(n,v,color=c,edgecolor='darkblue')
    axes[1,idx].set_ylim(0,1.05); axes[1,idx].set_title(f'{label} Metrics', fontweight='bold')
    for b,v in zip(bars,v): axes[1,idx].text(b.get_x()+b.get_width()/2,b.get_height()+0.02,f'{v:.4f}',ha='center',fontsize=8,fontweight='bold')

plt.tight_layout(); plt.show()

In [ ]:
res_df = pd.DataFrame(results)
res_df = res_df.set_index('Ratio')
print("=" * 60)
print("SVM Linear — Performance Comparison Across Split Ratios")
print("=" * 60)
print(res_df.to_string())
print("\n")
print("Best Accuracy:", res_df['Acc'].idxmax(), "-", f"{res_df['Acc'].max():.4f}")
print("Best AUC:     ", res_df['AUC'].idxmax(), "-", f"{res_df['AUC'].max():.4f}")